# Step 3E.1 - Hybrid Gate Actual Decode

This notebook validates the Step 3E.0 replay result with actual LLaDA decoding.

Selected gate:

```text
hybrid_gate_v1 = subject_match AND (
  relation_sim_rewrite >= 0.45
  OR relation_sim_bank >= 0.10
)
```

Relation bank source:

```text
runs/counterfact_direction1_v1/protocol/dev_tune_200.jsonl
```

This remains dev-only. Do not run `analysis_500` or final-test splits here.


In [ ]:
%%bash
set -euo pipefail

pip install -q \
  "transformers==4.46.3" \
  "datasets>=4.0.0" \
  "accelerate>=1.11.0" \
  "sentencepiece>=0.2.0" \
  "packaging" \
  "bitsandbytes>=0.43.0"


In [ ]:
from google.colab import drive

drive.mount('/content/drive')


## Environment And Paths


In [ ]:
from pathlib import Path
import csv
import json
import os
import platform
import subprocess
import sys
from collections import Counter, defaultdict

import torch
import transformers
import datasets
import accelerate

PROJECT_DIR = Path("/content/drive/MyDrive/Masters/SB V2/SB")
RUN_ROOT = PROJECT_DIR / "runs/counterfact_direction1_v1"
OUT_REPORT = RUN_ROOT / "dev_tune_200_hybrid_gate_decode_v1"

PROTOCOL_VERSION = "counterfact_direction1_v1"
GATE_ID = "hybrid_or_rel0.45_bank0.10"
GATE_MODE = "hybrid_relation_or"
RELATION_SIM_REWRITE_THRESHOLD = 0.45
RELATION_SIM_BANK_THRESHOLD = 0.10
RELATION_BANK_PATH = RUN_ROOT / "protocol/dev_tune_200.jsonl"
RELATION_BANK_SOURCE = "frozen_dev_tune_200_rewrite_templates"
ALLOW_OVERWRITE = False

print("python:", sys.version)
print("platform:", platform.platform())
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("accelerate:", accelerate.__version__)
print("project dir:", PROJECT_DIR)

try:
    commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=PROJECT_DIR, text=True).strip()
except Exception as exc:
    commit = None
    print("git commit unavailable:", repr(exc))
print("git commit:", commit)

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
from llada_experiment_reports import harmonic_mean, paired_bootstrap_delta_by_case
from llada_runtime_editor_eval import RolloutConfig, decompose_method, gate_should_activate


## Preflight


In [ ]:
DEV_EDITS = RUN_ROOT / "protocol/dev_tune_200.jsonl"
STRESS_INPUT = RUN_ROOT / "same_subject_stress_inputs/dev_tune_200_same_subject_stress.jsonl"
STRESS_SUMMARY = RUN_ROOT / "same_subject_stress_inputs/summary.json"
REPLAY_REPORT = RUN_ROOT / "dev_tune_200_hybrid_gate_replay_v1"
THRESHOLDS = RUN_ROOT / "dev_tune_200_runtime_baseline_thresholds.json"

required = [
    DEV_EDITS,
    STRESS_INPUT,
    STRESS_SUMMARY,
    RELATION_BANK_PATH,
    THRESHOLDS,
    REPLAY_REPORT / "report_summary.json",
    REPLAY_REPORT / "best_gate_candidates.csv",
    REPLAY_REPORT / "gate_replay_selection.csv",
]
for path in required:
    assert path.exists(), f"Missing required artifact: {path}"

with (REPLAY_REPORT / "report_summary.json").open() as f:
    replay_summary = json.load(f)
assert replay_summary["protocol_version"] == PROTOCOL_VERSION
assert replay_summary["analysis_500_used"] is False
assert replay_summary["final_test_used"] is False
assert replay_summary["uses_outcome_derived_advantage"] is False
assert GATE_ID in (REPLAY_REPORT / "gate_replay_selection.csv").read_text(), f"Missing replay gate_id {GATE_ID}"

cfg = RolloutConfig(
    steps=4,
    bridge_topk=4,
    mc_rollouts=2,
    guidance_scale=1.0,
    reward_mode="soft_overlap",
    reward_beta=6.0,
    target_logit_bias=0.0,
    gate_mode=GATE_MODE,
    temperature=1.0,
    relation_sim_rewrite_threshold=RELATION_SIM_REWRITE_THRESHOLD,
    relation_sim_bank_threshold=RELATION_SIM_BANK_THRESHOLD,
    relation_bank_path=str(RELATION_BANK_PATH),
    relation_bank_source=RELATION_BANK_SOURCE,
)
assert decompose_method("mc_bridge_gated_hybrid", cfg) == ("mc_bridge", GATE_MODE, "mc_bridge_gated_hybrid")

sample_edit = next(json.loads(line) for line in DEV_EDITS.open() if line.strip())
assert isinstance(gate_should_activate(sample_edit, sample_edit["prompt"], GATE_MODE, cfg), bool)

if OUT_REPORT.exists() and not ALLOW_OVERWRITE:
    raise RuntimeError(
        f"Report directory already exists: {OUT_REPORT}. "
        "Delete it manually, choose a new report name, or set ALLOW_OVERWRITE=True."
    )
print("Preflight OK.")


## Candidate Registry


In [ ]:
RUN_SPECS = [
    {
        "label": "prompt_memory_gated_hybrid",
        "method": "prompt_memory_gated_hybrid",
        "family": "prompt_memory",
        "guidance_scale": 1.0,
        "mc_rollouts": 0,
        "normal_dir": "dev_tune_200_hybrid_decode_prompt_memory",
        "stress_dir": "dev_tune_200_hybrid_decode_stress_prompt_memory",
    },
    {
        "label": "myopic_score_gated_hybrid_gs1.75",
        "method": "myopic_score_gated_hybrid",
        "family": "myopic_score",
        "guidance_scale": 1.75,
        "mc_rollouts": 0,
        "normal_dir": "dev_tune_200_hybrid_decode_myopic_gs175",
        "stress_dir": "dev_tune_200_hybrid_decode_stress_myopic_gs175",
    },
    {
        "label": "myopic_score_gated_hybrid_gs2.0",
        "method": "myopic_score_gated_hybrid",
        "family": "myopic_score",
        "guidance_scale": 2.0,
        "mc_rollouts": 0,
        "normal_dir": "dev_tune_200_hybrid_decode_myopic_gs200",
        "stress_dir": "dev_tune_200_hybrid_decode_stress_myopic_gs200",
    },
    {
        "label": "mc_bridge_gated_hybrid_gs1.75",
        "method": "mc_bridge_gated_hybrid",
        "family": "mc_bridge",
        "guidance_scale": 1.75,
        "mc_rollouts": 2,
        "normal_dir": "dev_tune_200_hybrid_decode_mc_bridge_gs175",
        "stress_dir": "dev_tune_200_hybrid_decode_stress_mc_bridge_gs175",
    },
    {
        "label": "mc_bridge_gated_hybrid_gs2.0",
        "method": "mc_bridge_gated_hybrid",
        "family": "mc_bridge",
        "guidance_scale": 2.0,
        "mc_rollouts": 2,
        "normal_dir": "dev_tune_200_hybrid_decode_mc_bridge_gs200",
        "stress_dir": "dev_tune_200_hybrid_decode_stress_mc_bridge_gs200",
    },
    {
        "label": "no_rollout_bridge_gated_hybrid_gs2.0",
        "method": "no_rollout_bridge_gated_hybrid",
        "family": "no_rollout_bridge",
        "guidance_scale": 2.0,
        "mc_rollouts": 0,
        "normal_dir": "dev_tune_200_hybrid_decode_no_rollout_gs200",
        "stress_dir": "dev_tune_200_hybrid_decode_stress_no_rollout_gs200",
    },
]
print(json.dumps(RUN_SPECS, indent=2))


## Overwrite Guard


In [ ]:
for spec in RUN_SPECS:
    for key in ["normal_dir", "stress_dir"]:
        path = RUN_ROOT / spec[key]
        if path.exists() and not ALLOW_OVERWRITE:
            raise RuntimeError(f"Run directory already exists: {path}")
if OUT_REPORT.exists() and not ALLOW_OVERWRITE:
    raise RuntimeError(f"Report directory already exists: {OUT_REPORT}")
print("Overwrite guard passed.")


## Run Actual Decoding

This runs each selected method on both normal `dev_tune_200` prompts and the same-subject stress input.


In [ ]:
def run_eval(spec, *, stress):
    run_dir = RUN_ROOT / (spec["stress_dir"] if stress else spec["normal_dir"])
    run_dir.mkdir(parents=True, exist_ok=False)
    edits_path = STRESS_INPUT if stress else DEV_EDITS
    cmd = [
        sys.executable,
        "-u",
        "llada_runtime_editor_eval.py",
        "--model_id", "GSAI-ML/LLaDA-8B-Base",
        "--edits_path", str(edits_path.relative_to(PROJECT_DIR)),
        "--output_dir", str(run_dir.relative_to(PROJECT_DIR)),
        "--methods", spec["method"],
        "--split_role", "dev_tune_200",
        "--protocol_version", PROTOCOL_VERSION,
        "--edit_access", "given_at_edit_time",
        "--training_access", "none",
        "--hyperparameter_access", "dev_tune_only",
        "--analysis_500_used", "0",
        "--final_test_used", "0",
        "--gate_mode", GATE_MODE,
        "--relation_sim_rewrite_threshold", str(RELATION_SIM_REWRITE_THRESHOLD),
        "--relation_sim_bank_threshold", str(RELATION_SIM_BANK_THRESHOLD),
        "--relation_bank_path", str(RELATION_BANK_PATH.relative_to(PROJECT_DIR)),
        "--relation_bank_source", RELATION_BANK_SOURCE,
        "--guidance_scale", str(spec["guidance_scale"]),
        "--bridge_topk", "4",
        "--mc_rollouts", str(spec["mc_rollouts"]),
        "--steps", "4",
        "--eval_samples", "4",
        "--reward_mode", "soft_overlap",
        "--reward_beta", "6.0",
        "--target_logit_bias", "0.0",
        "--seed", "0",
        "--use_4bit", "1",
        "--dtype", "float16",
        "--skip_candidate_coverage", "1",
    ]
    if stress:
        cmd.extend([
            "--stress_eval", "1",
            "--stress_name", "same_subject_locality_hybrid_decode",
            "--target_semantics", "target_new_over_injection",
        ])
    log_path = run_dir / "stdout.log"
    print("Running:", spec["label"], "stress" if stress else "normal")
    print(" ".join(cmd))
    with log_path.open("w", encoding="utf-8") as log:
        proc = subprocess.run(cmd, cwd=PROJECT_DIR, stdout=log, stderr=subprocess.STDOUT, text=True)
    if proc.returncode != 0:
        print(log_path.read_text()[-4000:])
        raise RuntimeError(f"Run failed: {spec['label']} stress={stress}; see {log_path}")
    print("Wrote:", run_dir)

for spec in RUN_SPECS:
    run_eval(spec, stress=False)
    run_eval(spec, stress=True)

print("All actual hybrid decode runs completed.")


## Config And Completeness Validation


In [ ]:
def read_jsonl(path):
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                yield json.loads(line)


def stress_bucket(prompt_id):
    prompt_id = str(prompt_id or "")
    if "_same_subject_template_" in prompt_id:
        return "same_subject_template"
    if "_generation_" in prompt_id:
        return "generation"
    if "_attribute_" in prompt_id:
        return "attribute"
    return "unknown"


def method_rows(run_dir, method, label, *, stress):
    rows = []
    for row in read_jsonl(run_dir / "per_case_results.jsonl"):
        method_name = row.get("method_variant") or row.get("method")
        if method_name != method:
            continue
        bucket = stress_bucket(row.get("prompt_id")) if stress else row.get("bucket")
        if bucket == "unknown":
            continue
        rows.append({**row, "method_variant": label, "bucket": bucket})
    return rows


def validate_run(spec, *, stress):
    run_dir = RUN_ROOT / (spec["stress_dir"] if stress else spec["normal_dir"])
    for artifact in ["run_config.json", "summary.json", "per_case_results.jsonl", "stdout.log"]:
        assert (run_dir / artifact).exists(), f"Missing {artifact}: {run_dir}"
    cfg = json.load((run_dir / "run_config.json").open())
    assert cfg["protocol_version"] == PROTOCOL_VERSION
    assert cfg["split_role"] == "dev_tune_200"
    assert cfg["analysis_500_used"] is False
    assert cfg["final_test_used"] is False
    assert cfg["methods"] == [spec["method"]]
    rollout = cfg["rollout_config"]
    assert rollout["gate_mode"] == GATE_MODE
    assert abs(float(rollout["relation_sim_rewrite_threshold"]) - RELATION_SIM_REWRITE_THRESHOLD) < 1e-8
    assert abs(float(rollout["relation_sim_bank_threshold"]) - RELATION_SIM_BANK_THRESHOLD) < 1e-8
    assert rollout["relation_bank_source"] == RELATION_BANK_SOURCE
    assert abs(float(rollout["guidance_scale"]) - float(spec["guidance_scale"])) < 1e-8
    assert int(rollout["mc_rollouts"]) == int(spec["mc_rollouts"])
    if stress:
        assert cfg["stress_eval"] is True
        assert cfg["stress_name"] == "same_subject_locality_hybrid_decode"
        assert cfg["target_semantics"] == "target_new_over_injection"
    rows = method_rows(run_dir, spec["method"], spec["label"], stress=stress)
    by_bucket = defaultdict(list)
    for row in rows:
        by_bucket[row["bucket"]].append(row)
    if stress:
        expected = {"same_subject_template": (200, 600), "generation": (200, 400)}
    else:
        expected = {
            "rewrite": (200, None),
            "declarative_paraphrases": (200, None),
            "near_locality": (200, None),
            "far_locality": (200, None),
        }
    for bucket, (expected_edits, expected_rows) in expected.items():
        bucket_rows = by_bucket[bucket]
        edits = {str(row.get("edit_id") or row.get("case_id")) for row in bucket_rows}
        assert len(edits) == expected_edits, (spec["label"], bucket, len(edits))
        if expected_rows is not None:
            assert len(bucket_rows) == expected_rows, (spec["label"], bucket, len(bucket_rows))
    print("Config and completeness OK:", spec["label"], "stress" if stress else "normal")

for spec in RUN_SPECS:
    validate_run(spec, stress=False)
    validate_run(spec, stress=True)
print("All actual hybrid decode configs and rows OK.")


## Build Actual Decode Report


In [ ]:
OUT_REPORT.mkdir(parents=True, exist_ok=False)

thresholds = json.load(THRESHOLDS.open())
selfloc_base = float(thresholds["selfloc_base"])
near_tfpr_budget = float(thresholds["target_false_positive_rate_budget_by_bucket"]["near_locality"])
far_tfpr_budget = float(thresholds["target_false_positive_rate_budget_by_bucket"]["far_locality"])
malformed_budget = float(thresholds["malformed_span_rate_budget"])
gpu_budget = float(thresholds["gpu_minutes_per_edit_budget"])

MIN_REWRITE_EXACT = 0.25
MIN_DECLARATIVE_PARAPHRASE_EXACT = 0.20
MIN_SELECTION_SCORE = 0.30
DRIFT_TOLERANCE = 0.03


def write_csv(path, rows):
    if not rows:
        path.write_text("", encoding="utf-8")
        return
    fieldnames = []
    for row in rows:
        for key in row:
            if key not in fieldnames:
                fieldnames.append(key)
    with path.open("w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def mean_or_none(values):
    vals = [float(v) for v in values if v is not None]
    return sum(vals) / len(vals) if vals else None


def aggregate(rows):
    by_bucket_edit = defaultdict(list)
    for row in rows:
        by_bucket_edit[(row["bucket"], str(row.get("edit_id") or row.get("case_id")))].append(row)
    by_bucket = defaultdict(list)
    for (bucket, edit_id), items in by_bucket_edit.items():
        by_bucket[bucket].append({
            "edit_id": edit_id,
            "exact_rate": mean_or_none(row.get("exact_rate") for row in items),
            "token_f1": mean_or_none(row.get("token_f1") for row in items),
            "malformed_rate": mean_or_none(row.get("malformed_rate") for row in items),
            "target_false_positive_rate": mean_or_none(row.get("target_false_positive_rate") for row in items),
            "sparse_guidance_kl": mean_or_none(row.get("sparse_guidance_kl") for row in items),
            "gate_activation_rate": mean_or_none(row.get("gate_activation_rate") for row in items),
            "num_prompt_rows": len(items),
            "target_length_bin": str(items[0].get("target_length_bin")),
        })
    summary = {}
    length_rows = []
    for bucket, items in by_bucket.items():
        summary[bucket] = {
            "num_edits": len(items),
            "num_prompt_rows": sum(row["num_prompt_rows"] for row in items),
            "mean_exact_rate": mean_or_none(row.get("exact_rate") for row in items),
            "mean_token_f1": mean_or_none(row.get("token_f1") for row in items),
            "mean_malformed_rate": mean_or_none(row.get("malformed_rate") for row in items),
            "mean_target_false_positive_rate": mean_or_none(row.get("target_false_positive_rate") for row in items),
            "mean_sparse_guidance_kl": mean_or_none(row.get("sparse_guidance_kl") for row in items),
            "gate_activation_rate": mean_or_none(row.get("gate_activation_rate") for row in items),
        }
        by_len = defaultdict(list)
        for row in items:
            by_len[str(row.get("target_length_bin"))].append(row)
        for length_bin, length_items in sorted(by_len.items()):
            length_rows.append({
                "bucket": bucket,
                "target_length_bin": length_bin,
                "num_edits": len(length_items),
                "mean_exact_rate": mean_or_none(row.get("exact_rate") for row in length_items),
                "mean_token_f1": mean_or_none(row.get("token_f1") for row in length_items),
                "mean_malformed_rate": mean_or_none(row.get("malformed_rate") for row in length_items),
                "mean_target_false_positive_rate": mean_or_none(row.get("target_false_positive_rate") for row in length_items),
                "mean_sparse_guidance_kl": mean_or_none(row.get("sparse_guidance_kl") for row in length_items),
            })
    return summary, length_rows

actual_rows_by_label = {}
method_bucket_rows = []
stress_rows = []
selection_rows = []
activation_rows = []
target_length_rows = []

# Stress budgets are actual base budgets from Step 3D report, via Step 3E.0 replay report.
replay_summary = json.load((REPLAY_REPORT / "report_summary.json").open())
same_subject_tfpr_budget = float(replay_summary["budgets"]["same_subject_template_tfpr"])
generation_tfpr_budget = float(replay_summary["budgets"]["generation_tfpr"])
stress_malformed_budget = float(replay_summary["budgets"].get("stress_malformed_rate", malformed_budget))

for spec in RUN_SPECS:
    normal_dir = RUN_ROOT / spec["normal_dir"]
    stress_dir = RUN_ROOT / spec["stress_dir"]
    rows = method_rows(normal_dir, spec["method"], spec["label"], stress=False)
    rows.extend(method_rows(stress_dir, spec["method"], spec["label"], stress=True))
    actual_rows_by_label[spec["label"]] = rows
    summary, length_rows = aggregate(rows)

    rewrite = summary["rewrite"].get("mean_exact_rate")
    paraphrase = summary["declarative_paraphrases"].get("mean_exact_rate")
    qa = summary.get("qa_format_generalization", {}).get("mean_exact_rate")
    locality = mean_or_none([
        summary["near_locality"].get("mean_exact_rate"),
        summary["far_locality"].get("mean_exact_rate"),
    ])
    clipped = min((locality or 0.0) / max(selfloc_base, 1e-6), 1.0)
    score = harmonic_mean([rewrite or 0.0, paraphrase or 0.0, clipped])
    near_tfpr = summary["near_locality"].get("mean_target_false_positive_rate")
    far_tfpr = summary["far_locality"].get("mean_target_false_positive_rate")
    same_subject_tfpr = summary["same_subject_template"].get("mean_target_false_positive_rate")
    generation_tfpr = summary["generation"].get("mean_target_false_positive_rate")
    max_malformed = max((bucket.get("mean_malformed_rate") or 0.0) for bucket in summary.values())
    primary_kl = mean_or_none([
        summary["rewrite"].get("mean_sparse_guidance_kl"),
        summary["declarative_paraphrases"].get("mean_sparse_guidance_kl"),
    ])
    gpu_minutes = None
    normal_eff = json.load((normal_dir / "summary.json").open()).get("efficiency", {})
    stress_eff = json.load((stress_dir / "summary.json").open()).get("efficiency", {})
    total_wall = float(normal_eff.get("wall_time_seconds") or 0.0) + float(stress_eff.get("wall_time_seconds") or 0.0)
    if total_wall:
        gpu_minutes = total_wall / 60.0 / 200.0

    near_pass = near_tfpr is not None and near_tfpr <= near_tfpr_budget
    far_pass = far_tfpr is not None and far_tfpr <= far_tfpr_budget
    same_subject_pass = same_subject_tfpr is not None and same_subject_tfpr <= same_subject_tfpr_budget
    generation_pass = generation_tfpr is not None and generation_tfpr <= generation_tfpr_budget
    malformed_pass = max_malformed <= malformed_budget and max_malformed <= stress_malformed_budget
    compute_pass = gpu_minutes is not None and gpu_minutes <= gpu_budget
    activation_pass = (
        (summary["rewrite"].get("gate_activation_rate") or 0.0) >= 0.95
        and (summary["declarative_paraphrases"].get("gate_activation_rate") or 0.0) >= 0.85
    )
    edit_useful_pass = rewrite >= MIN_REWRITE_EXACT and paraphrase >= MIN_DECLARATIVE_PARAPHRASE_EXACT and score >= MIN_SELECTION_SCORE
    template_stress_pass = same_subject_pass and malformed_pass and activation_pass
    strict_stress_pass = same_subject_pass and generation_pass and malformed_pass and activation_pass
    constraint_pass = near_pass and far_pass and same_subject_pass and malformed_pass and compute_pass
    decision_status = "keep_dev_candidate" if constraint_pass and edit_useful_pass and activation_pass else "reject_or_diagnostic"

    selection = {
        "label": spec["label"],
        "family": spec["family"],
        "guidance_scale": spec["guidance_scale"],
        "rewrite_exact": rewrite,
        "declarative_paraphrases_exact": paraphrase,
        "qa_format_generalization_exact": qa,
        "locality_exact": locality,
        "selfloc_base": selfloc_base,
        "clipped_self_normalized_locality": clipped,
        "selection_score": score,
        "primary_sparse_guidance_kl": primary_kl,
        "near_locality_tfpr": near_tfpr,
        "far_locality_tfpr": far_tfpr,
        "same_subject_template_tfpr": same_subject_tfpr,
        "generation_tfpr": generation_tfpr,
        "max_malformed_rate": max_malformed,
        "gpu_minutes_per_edit": gpu_minutes,
        "near_tfpr_pass": near_pass,
        "far_tfpr_pass": far_pass,
        "same_subject_template_tfpr_pass": same_subject_pass,
        "generation_tfpr_pass": generation_pass,
        "malformed_pass": malformed_pass,
        "compute_pass": compute_pass,
        "activation_pass": activation_pass,
        "template_stress_pass": template_stress_pass,
        "strict_stress_pass": strict_stress_pass,
        "edit_useful_pass": edit_useful_pass,
        "constraint_pass": constraint_pass,
        "decision_status": decision_status,
        "constraint_violations": ";".join(name for name, passed in [
            ("near_tfpr", near_pass),
            ("far_tfpr", far_pass),
            ("same_subject_template_tfpr", same_subject_pass),
            ("malformed", malformed_pass),
            ("compute", compute_pass),
            ("activation", activation_pass),
            ("edit_useful", edit_useful_pass),
        ] if not passed),
    }
    selection_rows.append(selection)

    for bucket, bucket_summary in sorted(summary.items()):
        row = {"label": spec["label"], "family": spec["family"], "guidance_scale": spec["guidance_scale"], "bucket": bucket, **bucket_summary}
        method_bucket_rows.append(row)
        activation_rows.append({
            "label": spec["label"],
            "family": spec["family"],
            "guidance_scale": spec["guidance_scale"],
            "bucket": bucket,
            "gate_activation_rate": bucket_summary.get("gate_activation_rate"),
            "num_edits": bucket_summary.get("num_edits"),
            "num_prompt_rows": bucket_summary.get("num_prompt_rows"),
        })
        if bucket in {"same_subject_template", "generation"}:
            stress_rows.append(row)
    for row in length_rows:
        target_length_rows.append({"label": spec["label"], "family": spec["family"], "guidance_scale": spec["guidance_scale"], **row})

selection_rows.sort(key=lambda row: (row["decision_status"] != "keep_dev_candidate", -(row["selection_score"] or 0.0)))

write_csv(OUT_REPORT / "hybrid_gate_method_bucket.csv", method_bucket_rows)
write_csv(OUT_REPORT / "hybrid_gate_stress_summary.csv", stress_rows)
write_csv(OUT_REPORT / "hybrid_gate_selection.csv", selection_rows)
write_csv(OUT_REPORT / "hybrid_gate_gate_activation_summary.csv", activation_rows)
write_csv(OUT_REPORT / "hybrid_gate_target_length.csv", target_length_rows)

print("Selection rows:")
for row in selection_rows:
    print(row["decision_status"], row["label"], row["selection_score"], row["constraint_violations"])


## Replay vs Actual Comparison


In [ ]:
replay_rows = []
with (REPLAY_REPORT / "gate_replay_selection.csv").open(newline="", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        if row.get("gate_id") == GATE_ID:
            replay_rows.append(row)
replay_by_method = {row["replay_method_label"]: row for row in replay_rows}
actual_by_method = {row["label"]: row for row in selection_rows}

METRIC_MAP = [
    ("rewrite_exact", "rewrite_exact"),
    ("declarative_paraphrases_exact", "declarative_paraphrases_exact"),
    ("same_subject_template_tfpr", "same_subject_template_tfpr"),
    ("generation_tfpr", "generation_tfpr"),
    ("rewrite_gate_activation", "rewrite_gate_activation"),
    ("declarative_paraphrases_gate_activation", "declarative_paraphrases_gate_activation"),
    ("same_subject_template_gate_activation", "same_subject_template_gate_activation"),
]

def as_float(value):
    if value in {None, ""}:
        return None
    return float(value)

comparison_rows = []
for spec in RUN_SPECS:
    label = spec["label"]
    replay = replay_by_method.get(label)
    actual = actual_by_method[label]
    assert replay is not None, f"Missing replay row for {label} / {GATE_ID}"
    row = {"label": label, "family": spec["family"], "guidance_scale": spec["guidance_scale"], "gate_id": GATE_ID}
    drift_failures = []
    for replay_key, actual_key in METRIC_MAP:
        r = as_float(replay.get(replay_key))
        a = as_float(actual.get(actual_key))
        drift = None if r is None or a is None else a - r
        passed = True if drift is None else abs(drift) <= DRIFT_TOLERANCE
        if not passed:
            drift_failures.append(replay_key)
        row[f"{replay_key}_replay"] = r
        row[f"{actual_key}_actual"] = a
        row[f"{actual_key}_drift"] = drift
        row[f"{actual_key}_drift_pass"] = passed
    row["all_drift_pass"] = not drift_failures
    row["drift_failures"] = ";".join(drift_failures)
    comparison_rows.append(row)

write_csv(OUT_REPORT / "replay_vs_actual_comparison.csv", comparison_rows)
print("Replay vs actual:")
for row in comparison_rows:
    print(row["label"], "pass" if row["all_drift_pass"] else "DRIFT", row["drift_failures"])


## Paired Bootstrap And Output Samples


In [ ]:
bootstrap_pairs = [
    ("myopic_score_gated_hybrid_gs2.0", "mc_bridge_gated_hybrid_gs2.0"),
    ("myopic_score_gated_hybrid_gs2.0", "prompt_memory_gated_hybrid"),
    ("mc_bridge_gated_hybrid_gs2.0", "mc_bridge_gated_hybrid_gs1.75"),
    ("mc_bridge_gated_hybrid_gs2.0", "no_rollout_bridge_gated_hybrid_gs2.0"),
    ("mc_bridge_gated_hybrid_gs2.0", "prompt_memory_gated_hybrid"),
]
bootstrap_rows = []
for candidate, baseline in bootstrap_pairs:
    pair_rows = actual_rows_by_label[candidate] + actual_rows_by_label[baseline]
    for bucket, metric_name, metric in [
        ("rewrite", "rewrite_exact", "exact_rate"),
        ("declarative_paraphrases", "declarative_paraphrases_exact", "exact_rate"),
        ("same_subject_template", "same_subject_template_tfpr", "target_false_positive_rate"),
        ("generation", "generation_tfpr", "target_false_positive_rate"),
        ("near_locality", "near_tfpr", "target_false_positive_rate"),
        ("far_locality", "far_tfpr", "target_false_positive_rate"),
    ]:
        boot = paired_bootstrap_delta_by_case(
            pair_rows,
            candidate_method=candidate,
            baseline_method=baseline,
            bucket=bucket,
            metric=metric,
            samples=1000,
            seed=0,
        )
        if boot:
            bootstrap_rows.append({
                "candidate": candidate,
                "baseline": baseline,
                "bucket": bucket,
                "metric": metric_name,
                "mean_delta_candidate_minus_baseline": boot["mean_delta"],
                "ci_lower": boot["ci_low"],
                "ci_upper": boot["ci_high"],
                "num_edits": boot["num_edits"],
            })
write_csv(OUT_REPORT / "hybrid_gate_paired_bootstrap.csv", bootstrap_rows)

sample_rows = []
for label, rows in actual_rows_by_label.items():
    groups = defaultdict(list)
    for row in rows:
        if row["bucket"] == "same_subject_template" and (row.get("target_false_positive_rate") or 0.0) > 0.0:
            groups["same_subject_leakage"].append(row)
        if row["bucket"] == "same_subject_template" and (row.get("gate_activation_rate") or 0.0) == 0.0:
            groups["same_subject_gate_off"].append(row)
        if row["bucket"] in {"rewrite", "declarative_paraphrases"} and (row.get("gate_activation_rate") or 0.0) == 0.0:
            groups["edit_prompt_gate_off"].append(row)
        if (row.get("malformed_rate") or 0.0) > 0.0:
            groups["malformed"].append(row)
    for category, items in groups.items():
        for row in items[:12]:
            sample_rows.append({
                "sample_category": category,
                "label": label,
                "edit_id": row.get("edit_id"),
                "case_id": row.get("case_id"),
                "relation_id": row.get("relation_id"),
                "target_length_bin": row.get("target_length_bin"),
                "bucket": row.get("bucket"),
                "prompt_id": row.get("prompt_id"),
                "prompt": row.get("prompt"),
                "target": row.get("target"),
                "gate_activation_rate": row.get("gate_activation_rate"),
                "exact_rate": row.get("exact_rate"),
                "target_false_positive_rate": row.get("target_false_positive_rate"),
                "malformed_rate": row.get("malformed_rate"),
                "sample_outputs": json.dumps(row.get("sample_outputs", []), ensure_ascii=False),
                "greedy_output": row.get("greedy_output"),
            })
write_csv(OUT_REPORT / "hybrid_gate_output_samples.csv", sample_rows)
print("bootstrap rows:", len(bootstrap_rows))
print("sample rows:", len(sample_rows))


## Report Summary


In [ ]:
report = {
    "protocol_version": PROTOCOL_VERSION,
    "split_role": "dev_tune_200",
    "analysis_500_used": False,
    "final_test_used": False,
    "stage": "Step 3E.1 hybrid gate actual decode",
    "gate_id": GATE_ID,
    "gate_mode": GATE_MODE,
    "relation_sim_rewrite_threshold": RELATION_SIM_REWRITE_THRESHOLD,
    "relation_sim_bank_threshold": RELATION_SIM_BANK_THRESHOLD,
    "relation_bank_path": str(RELATION_BANK_PATH.relative_to(PROJECT_DIR)),
    "relation_bank_source": RELATION_BANK_SOURCE,
    "primary_stress_bucket": "same_subject_template",
    "secondary_stress_bucket": "generation",
    "qa_format_generalization": "secondary_diagnostic",
    "selfloc_base": selfloc_base,
    "drift_tolerance": DRIFT_TOLERANCE,
    "candidate_labels": [spec["label"] for spec in RUN_SPECS],
    "budgets": {
        "near_locality_tfpr": near_tfpr_budget,
        "far_locality_tfpr": far_tfpr_budget,
        "same_subject_template_tfpr": same_subject_tfpr_budget,
        "generation_tfpr": generation_tfpr_budget,
        "malformed_rate": malformed_budget,
        "stress_malformed_rate": stress_malformed_budget,
        "gpu_minutes_per_edit": gpu_budget,
    },
    "edit_useful_thresholds": {
        "min_rewrite_exact": MIN_REWRITE_EXACT,
        "min_declarative_paraphrases_exact": MIN_DECLARATIVE_PARAPHRASE_EXACT,
        "min_selection_score": MIN_SELECTION_SCORE,
    },
    "artifacts": {
        "method_bucket": str(OUT_REPORT / "hybrid_gate_method_bucket.csv"),
        "stress_summary": str(OUT_REPORT / "hybrid_gate_stress_summary.csv"),
        "selection": str(OUT_REPORT / "hybrid_gate_selection.csv"),
        "paired_bootstrap": str(OUT_REPORT / "hybrid_gate_paired_bootstrap.csv"),
        "target_length": str(OUT_REPORT / "hybrid_gate_target_length.csv"),
        "gate_activation": str(OUT_REPORT / "hybrid_gate_gate_activation_summary.csv"),
        "output_samples": str(OUT_REPORT / "hybrid_gate_output_samples.csv"),
        "replay_vs_actual": str(OUT_REPORT / "replay_vs_actual_comparison.csv"),
    },
}
with (OUT_REPORT / "report_summary.json").open("w", encoding="utf-8") as f:
    json.dump(report, f, indent=2)

expected_files = [
    "report_summary.json",
    "hybrid_gate_method_bucket.csv",
    "hybrid_gate_stress_summary.csv",
    "hybrid_gate_selection.csv",
    "hybrid_gate_paired_bootstrap.csv",
    "hybrid_gate_target_length.csv",
    "hybrid_gate_gate_activation_summary.csv",
    "hybrid_gate_output_samples.csv",
    "replay_vs_actual_comparison.csv",
]
for name in expected_files:
    assert (OUT_REPORT / name).exists(), f"Missing report artifact: {name}"

print("Wrote:", OUT_REPORT)
print(json.dumps({"num_methods": len(RUN_SPECS), "gate_id": GATE_ID}, indent=2))


## Expected Decision

Inspect first:

```text
hybrid_gate_selection.csv
replay_vs_actual_comparison.csv
hybrid_gate_stress_summary.csv
hybrid_gate_gate_activation_summary.csv
hybrid_gate_paired_bootstrap.csv
hybrid_gate_output_samples.csv
```

If actual decoding matches replay and at least one method remains a dev candidate, the hybrid gate can be treated as the first viable dev gate.

Do not move to `analysis_500` yet. If this confirms replay, the next decision is whether to continue with:

```text
Track A: myopic_score_gated_hybrid_gs2.0 as best empirical runtime editor
Track B: mc_bridge_gated_hybrid_gs2.0 or gs1.75 as best bridge editor
```
